# Lab 3 : Keyword search vs embedding search on the same query

*W3 RAG Part 1 · Utrains LLMOps 8-Week Course*

Run each cell in order. Read the output. Move to the next.

See the matching slide in this week's concepts deck for the real-world story this lab teaches.

## What we are achieving in this lab

**Objective.** Search a pile of snippets with one question. Run two scores: **keyword** (shared words) and **cosine** (Lab 2). See which one finds the useful rows.

**Prerequisites.** Labs 1 and 2 finished. Same OpenAI key.

**What this lab uses.**

| Layer | What it does | What we use |
|-------|----------------|-------------|
| Keyword score | Shared words. No vectors. | Count overlapping words |
| Embedding score | Meaning | Cosine (Lab 2) |

**What you will do.**

1. Store six support snippets.
2. Ask a question that does **not** reuse the useful words.
3. Rank by keywords, then by cosine.

**Cost.** A few embedding calls. The keyword step does not call OpenAI.

## Where this sits after Labs 1 and 2

Lab 1: text becomes a vector.
Lab 2: two vectors get a cosine score.

Lab 3 is **search**. A user asks a question. You score every snippet and keep the top few.

Two scores for that same job:

- **Keyword search** — shared words. No embeddings. The older way (Ctrl+F, grep).
- **Embedding search** — Lab 1 + Lab 2 cosine. Meaning.

We run keyword first so you see the hole cosine fills: the customer says "sign-in details," the handbook says "credentials." Same meaning, different words. Cosine finds it. Keywords often do not.

Keyword search is still used in production RAG (the usual name is **BM25**) when the query is an exact token: an error code, a SKU. Hybrid means run both. We do not build hybrid today.

## The question we will ask on purpose

The snippets say **"credentials"** and **"Account > Profile"**.

We ask: *Where can users edit their sign-in details?*

Different words, same meaning. Keyword search should miss. Cosine should hit.

### Step 1. Same embeddings object as Labs 1 and 2

If the key cell fails, fix `.env` from Lab 1 first. Step 2 (keyword) will not use `embed` or `cosine`.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(
        "Missing OPENAI_API_KEY. Copy .env.example to .env at the repository root, "
        "paste the key, and restart the kernel."
    )

print("OPENAI_API_KEY : set")

OPENAI_API_KEY : set


In [2]:
import numpy as np
from langchain_openai import OpenAIEmbeddings

# Same model as Labs 1 and 2. Cosine is only legal inside one model.
EMBED_MODEL = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)


def embed(text: str) -> np.ndarray:
    # Lab 1: one string -> one vector.
    return np.asarray(embeddings.embed_query(text), dtype=float)


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    # Lab 2: how close two arrows are. Higher = closer in meaning.
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


CORPUS = [
    "To update your credentials, visit Account > Profile > Edit.",
    "Our offices open at 9am Pacific.",
    "If you forgot your password, click Forgot Password on the sign-in page.",
    "We accept Visa, Mastercard, and American Express.",
    "To change your email address, open Account > Security.",
    "Refunds are issued within 30 days under our standard policy.",
]

QUERY = "Where can users edit their sign-in details?"
print("Question:", QUERY)
print("Snippets stored:", len(CORPUS))

Question: Where can users edit their sign-in details?
Snippets stored: 6


### Step 2. Keyword search

Count how many words from the question also appear in each snippet.

No `embed`. No `cosine`. No OpenAI. If the words do not match, the score is zero.

Production keyword search is usually **BM25** (rarer words count more). Same hole on paraphrases. We do not implement BM25 here.

In [3]:
def words(text: str) -> list[str]:
    # Lowercase, strip punctuation, split on spaces.
    # No embed(), no cosine() — this path never leaves the words.
    cleaned = text.lower().replace(",", "").replace(".", "").replace("?", "").replace("!", "")
    return cleaned.split()


query_words = words(QUERY)
print("Words in the question:", query_words)
print()
print("matches = how many of those words also appear in the snippet")
print()

for doc in CORPUS:
    doc_words = words(doc)
    shared = []
    for w in query_words:
        if w in doc_words and w not in shared:
            shared.append(w)
    print("matches =", len(shared), " ", shared if shared else "-", " ", doc)

Words in the question: ['where', 'can', 'users', 'edit', 'their', 'sign-in', 'details']

matches = how many of those words also appear in the snippet

matches = 1   ['edit']   To update your credentials, visit Account > Profile > Edit.
matches = 0   -   Our offices open at 9am Pacific.
matches = 1   ['sign-in']   If you forgot your password, click Forgot Password on the sign-in page.
matches = 0   -   We accept Visa, Mastercard, and American Express.
matches = 0   -   To change your email address, open Account > Security.
matches = 0   -   Refunds are issued within 30 days under our standard policy.


The useful rows (credentials / Profile, email / Security) should show **0**, or a leftover like `"edit"`.

The question said "sign-in details." The docs said "credentials." Keywords cannot see that. Cosine can. That is why RAG embeds.

### Step 3. Embedding search (Lab 2 cosine)

Same question. Same snippets. Score with cosine. Highest first.

In [4]:
print("Question:", QUERY)
print("Score 1.0 = same direction. Lower = less related.")
print()

q_vec = embed(QUERY)

scored = []
for doc in CORPUS:
    score = cosine(q_vec, embed(doc))
    scored.append((score, doc))
    print(round(score, 3), " ", doc)

print()
print("Highest first:")
scored.sort(reverse=True)
for score, doc in scored:
    print(round(score, 3), " ", doc)

Question: Where can users edit their sign-in details?
Score 1.0 = same direction. Lower = less related.

0.551   To update your credentials, visit Account > Profile > Edit.
0.143   Our offices open at 9am Pacific.
0.423   If you forgot your password, click Forgot Password on the sign-in page.
0.183   We accept Visa, Mastercard, and American Express.
0.458   To change your email address, open Account > Security.
0.128   Refunds are issued within 30 days under our standard policy.

Highest first:
0.551   To update your credentials, visit Account > Profile > Edit.
0.458   To change your email address, open Account > Security.
0.423   If you forgot your password, click Forgot Password on the sign-in page.
0.183   We accept Visa, Mastercard, and American Express.
0.143   Our offices open at 9am Pacific.
0.128   Refunds are issued within 30 days under our standard policy.


The account-settings rows should be at the top. They shared meaning, not words.

Keywords miss paraphrases. Cosine can miss an exact token like `E4221` in a long page. Production RAG often runs **both** (hybrid). We do not fuse lists today.

## What you should be able to explain

> "Keyword search matches words. Embedding search matches meaning with cosine. I can show a query where one succeeds and the other fails."

> "Production keyword search is usually BM25. Hybrid is both keyword and cosine."

**Lab 4** is how you split a long document into the snippets you just searched.